# OptimUS quick reference


This notebook is a one-page reference for the public functions of OptimUS: what each one is called, what arguments it takes, the units, and the smallest example that runs. It is meant to be read alongside the other example notebooks, which explain each feature in depth.

All signatures below were checked against the OptimUS v0.2.1 source code in this repository. The code cells are written for the `optimuslib/optimus` Docker image (Python 3.5), so they avoid f-strings and other newer syntax.

### Where to go for more detail

| Topic | Section here | Example notebook with the full story |
|---|---|---|
| End-to-end workflow | 1 | `Basic model.ipynb` |
| Plane wave and point source | 3.1, 3.2 | `Basic model.ipynb` |
| Piston source | 3.3 | `Piston source.ipynb`, `Acoustic intensity and power calculations for a piston source.ipynb` |
| Bowl source | 3.4 | `Bowl source.ipynb` |
| Array source | 3.5 | `Array source configuration.ipynb`, `Array source simulation.ipynb` |
| Geometry and meshes | 4 | `Object shapes.ipynb` |
| Materials | 5 | `Material properties.ipynb` |
| Models and preconditioners | 6 | `Preconditioners.ipynb`, `Multiple domains.ipynb`, `Nested domains.ipynb` |
| Global parameters | 7 | `Global parameters.ipynb` |
| Postprocessing and plotting | 8 | `Visualisation and postprocess.ipynb`, `Visualisation in 3D.ipynb` |
| Export and import | 9 | `Visualisation and postprocess.ipynb` |

In [ ]:
import numpy as np
import optimus
print("OptimUS version:", optimus.__version__)

## 1. The workflow at a glance

Every OptimUS simulation follows the same five steps:

1. **Source**: `optimus.source.create_*` builds the incident field (plane wave, point source, piston, bowl or array).
2. **Geometry**: `optimus.geometry.shapes.*` or `optimus.geometry.load.import_grid` gives the surface mesh of each scatterer.
3. **Materials**: `optimus.material.load_material` or `create_material` gives the exterior and interior media.
4. **Model**: `optimus.model.create_default_model` (or `create_acoustic_model`) assembles the BEM system, then `model.solve()` solves it.
5. **Postprocess**: `optimus.postprocess.VisualisePlane` evaluates the field on a plane and `plot_pressure_field` draws it.

The cell below is the smallest complete simulation: a 1 kHz plane wave hitting a bone sphere of radius 1 m in water.

In [ ]:
frequency = 1000                                                      # Hz
source = optimus.source.create_planewave(frequency, direction=(1, 0, 0))
geometry = optimus.geometry.shapes.Sphere(radius=1, element_size=0.4) # m
material_ext = optimus.material.load_material("water")
material_int = optimus.material.load_material("bone (cortical)")

model = optimus.model.create_default_model(source, geometry, material_ext, material_int)
model.solve()
print("GMRES iterations:", model.iteration_count)

postprocess_plane = optimus.postprocess.VisualisePlane(model)
postprocess_plane.create_computational_grid(resolution=(101, 101), bounding_box=(-3, 3, -3, 3))
postprocess_plane.compute_fields()

In [ ]:
%matplotlib inline
figs = optimus.postprocess.plot_pressure_field(postprocess_plane, field="total")

## 2. Units and conventions

| Quantity | Unit | Notes |
|---|---|---|
| Frequency | Hz | Write 1 MHz as `1e6`. |
| Lengths, radii, locations, element size | m | A 32 mm radius is `32e-3`. |
| Pressure | Pa | `plot_pressure_field(..., unit="MPa")` only rescales the plot. |
| Normal velocity of a transducer | m/s | Can be complex, to set a phase. |
| Density | kg/m^3 | |
| Speed of sound | m/s | |
| Attenuation | Np/m/MHz^b | Power law `alpha = a * f_MHz ** b`, see section 5. |

Other conventions:

- **Points are 3xN arrays.** Any function that takes a set of locations expects shape `(3, N)`: row 0 is x, row 1 is y, row 2 is z.
- **Directions and axes are 3-tuples** such as `(1, 0, 0)`. They do not need to be normalised.
- **Time dependence** is harmonic; fields are complex amplitudes, so plot `np.real(p)` for a snapshot and `np.abs(p)` for the amplitude.
- **Mesh size rule of thumb:** use at least 6 elements per wavelength in the material with the shortest wavelength, so `element_size = wavelength / 6`.

## 3. Sources (`optimus.source`)

All source constructors take the `frequency` in Hz as their first argument and return a source object. Every source object has the same method to evaluate its incident field in a medium:

```python
p_inc = source.pressure_field(medium, locations)   # locations: (3, N) array, returns complex (N,) array
```

Transducer sources (piston, bowl, array) are discretised as a set of point sources; `number_of_point_sources_per_wavelength` controls how fine that is (default 6).

### 3.1 Plane wave

```python
optimus.source.create_planewave(frequency, direction=(1, 0, 0), amplitude=1.0)
```

| Argument | Type | Default | Meaning |
|---|---|---|---|
| `frequency` | float | required | Frequency in Hz |
| `direction` | 3-tuple | `(1, 0, 0)` | Propagation direction |
| `amplitude` | float | `1.0` | Pressure amplitude in Pa |

In [ ]:
water = optimus.material.load_material("water")
planewave = optimus.source.create_planewave(frequency=1000, direction=(1, 0, 0), amplitude=1.0)

# Evaluate along the x-axis: shape (3, N)
x = np.linspace(0, 3, 7)
points = np.vstack((x, np.zeros_like(x), np.zeros_like(x)))
print(np.round(planewave.pressure_field(water, points), 3))

### 3.2 Point source

```python
optimus.source.create_pointsource(frequency, location=(0, 0, 0), amplitude=1.0)
```

| Argument | Type | Default | Meaning |
|---|---|---|---|
| `frequency` | float | required | Frequency in Hz |
| `location` | 3-tuple | `(0, 0, 0)` | Position of the source in m |
| `amplitude` | float | `1.0` | Pressure amplitude in Pa at 1 m from the source |

In [ ]:
pointsource = optimus.source.create_pointsource(frequency=1000, location=(0, 0, 0), amplitude=1.0)
x = np.array([0.5, 1.0, 2.0])
points = np.vstack((x, np.zeros_like(x), np.zeros_like(x)))
print("|p| at 0.5, 1, 2 m:", np.round(np.abs(pointsource.pressure_field(water, points)), 3))

### 3.3 Piston

```python
optimus.source.create_piston(frequency, source_axis=(1, 0, 0),
                             number_of_point_sources_per_wavelength=6,
                             location=(0, 0, 0), velocity=1.0,
                             shape="circular", radius=None, width=None, height=None)
```

| Argument | Type | Default | Meaning |
|---|---|---|---|
| `frequency` | float | required | Frequency in Hz |
| `source_axis` | 3-tuple | `(1, 0, 0)` | Direction the piston faces |
| `number_of_point_sources_per_wavelength` | int | `6` | Discretisation of the piston surface |
| `location` | 3-tuple | `(0, 0, 0)` | Centre of the piston face in m |
| `velocity` | complex | `1.0` | Normal velocity in m/s |
| `shape` | str | `"circular"` | `"circular"` or `"rectangular"` |
| `radius` | float | `None` | Radius in m (circular shape, required) |
| `width`, `height` | float | `None` | Size in m (rectangular shape, required) |

In [ ]:
frequency = 500e3                                           # 500 kHz
piston = optimus.source.create_piston(frequency, radius=5e-3, location=(0, 0, 0),
                                      source_axis=(1, 0, 0), velocity=0.4)

# On-axis pressure magnitude from 1 mm to 50 mm
x = np.linspace(1e-3, 50e-3, 200)
points = np.vstack((x, np.zeros_like(x), np.zeros_like(x)))
p_axis = piston.pressure_field(water, points)
print("Peak on-axis |p| = %.3f MPa at x = %.1f mm" % (np.abs(p_axis).max() / 1e6, 1e3 * x[np.argmax(np.abs(p_axis))]))

### 3.4 Bowl (spherical section)

```python
optimus.source.create_bowl(frequency, outer_radius, radius_of_curvature,
                           source_axis=(1, 0, 0),
                           number_of_point_sources_per_wavelength=6,
                           location=(0, 0, 0), velocity=1.0, inner_radius=None)
```

| Argument | Type | Default | Meaning |
|---|---|---|---|
| `frequency` | float | required | Frequency in Hz |
| `outer_radius` | float | required | Aperture radius of the bowl in m |
| `radius_of_curvature` | float | required | Radius of the sphere the bowl is cut from, in m. The geometric focus is this distance along `source_axis` from the bowl apex. |
| `source_axis` | 3-tuple | `(1, 0, 0)` | Direction the bowl faces |
| `number_of_point_sources_per_wavelength` | int | `6` | Discretisation of the bowl surface |
| `location` | 3-tuple | `(0, 0, 0)` | Location of the bowl apex (centre of the curved surface) in m. The docstring says "centroid", but the source code places the apex here. |
| `velocity` | complex | `1.0` | Normal velocity in m/s |
| `inner_radius` | float | `None` | Radius of a central hole in m (for an imaging probe); `None` or `0` for no hole |

So with the defaults, a bowl with `radius_of_curvature=64e-3` focuses at `(64e-3, 0, 0)`.

In [ ]:
frequency = 500e3
bowl = optimus.source.create_bowl(frequency, outer_radius=32e-3, radius_of_curvature=64e-3,
                                  source_axis=(1, 0, 0), location=(0, 0, 0), velocity=0.04)

x = np.linspace(20e-3, 100e-3, 200)
points = np.vstack((x, np.zeros_like(x), np.zeros_like(x)))
p_axis = bowl.pressure_field(water, points)
print("Peak on-axis |p| = %.3f MPa at x = %.1f mm" % (np.abs(p_axis).max() / 1e6, 1e3 * x[np.argmax(np.abs(p_axis))]))

### 3.5 Array

```python
optimus.source.create_array(frequency, element_radius=None, element_width=None,
                            element_height=None, element_shape="circular",
                            velocity=1.0, source_axis=(1, 0, 0),
                            number_of_point_sources_per_wavelength=6,
                            location=(0, 0, 0), centroid_locations=None,
                            centroid_locations_filename=None, array_type="spherical")
```

| Argument | Type | Default | Meaning |
|---|---|---|---|
| `element_radius` | float | `None` | Radius of each circular element in m |
| `element_width`, `element_height` | float | `None` | Size of each rectangular element in m |
| `element_shape` | str | `"circular"` | `"circular"` or `"rectangular"` |
| `velocity` | complex or (N,) array | `1.0` | One value for all elements, or one per element (use this for phasing/steering) |
| `centroid_locations` | (3, N) array | `None` | Element centres, in the array's local frame |
| `centroid_locations_filename` | str | `None` | Alternative to `centroid_locations`: a file with the element centres |
| `array_type` | str | `"spherical"` | `"spherical"` (elements on a bowl) or `"planar"` |

**Element positions are given in a local frame**, not in global coordinates:

- `"spherical"`: the transducer axis is the local +z-axis and the **focus is the local origin**. Every centroid must lie on a sphere centred at the origin (the code checks this to within 1e-6 m and computes the radius of curvature from it), so the elements sit at negative z.
- `"planar"`: the centroids must lie in the local z = 0 plane and the elements face +z.

The array is then rotated so the local z-axis points along `source_axis` and moved so its **apex sits at `location`**, exactly like the bowl. A spherical array with radius of curvature R therefore focuses at `location + R * source_axis`.

The example below places 3 rings of elements on a 64 mm radius bowl.

In [ ]:
frequency = 500e3
R = 64e-3                                      # radius of curvature, m
centroids = []
for polar_deg, n_elements in ((10, 6), (20, 12), (30, 18)):
    polar = np.radians(polar_deg)
    for k in range(n_elements):
        phi = 2 * np.pi * k / n_elements
        centroids.append((R * np.sin(polar) * np.cos(phi),
                          R * np.sin(polar) * np.sin(phi),
                          -R * np.cos(polar)))   # negative z: the focus is at the origin
centroids = np.array(centroids).T              # shape (3, N)

array_source = optimus.source.create_array(frequency, element_radius=3e-3,
                                           centroid_locations=centroids,
                                           source_axis=(1, 0, 0), location=(0, 0, 0),
                                           velocity=0.04)
print("Number of elements:", centroids.shape[1])

x = np.linspace(20e-3, 100e-3, 200)
points = np.vstack((x, np.zeros_like(x), np.zeros_like(x)))
p_axis = array_source.pressure_field(water, points)
print("Peak on-axis |p| = %.3f MPa at x = %.1f mm" % (np.abs(p_axis).max() / 1e6, 1e3 * x[np.argmax(np.abs(p_axis))]))

To steer or apodise the beam, pass one complex velocity per element, for example `velocity=0.04 * np.exp(1j * phases)`. The element centres can also be read from a whitespace-separated text file with 3 rows (x, y, z) via `centroid_locations_filename="my_array.dat"`; the file must end in `.dat`, and you cannot pass both `centroid_locations` and a filename. `Focusing an array source using phase conjugation.ipynb` shows how to compute focusing phases.

## 4. Geometry (`optimus.geometry`)

A geometry is a closed triangular surface mesh. Built-in shapes live in `optimus.geometry.shapes`; all lengths are in m and `element_size` sets the mesh width.

| Constructor | Arguments (defaults) |
|---|---|
| `shapes.Sphere` | `radius=1, origin=(0, 0, 0), element_size=0.1, label="sphere"` |
| `shapes.Ellipsoid` | `radius=(1, 1, 1), origin=(0, 0, 0), element_size=0.1, label="ellipsoid"` |
| `shapes.Cube` | `length=1, origin=(0, 0, 0), element_size=0.1, label="cube"` |
| `shapes.Cuboid` | `length=(1, 1, 1), origin=(0, 0, 0), element_size=0.1, label="cuboid"` |
| `shapes.RegularSphere` | `refinement_level=1, label="regular_sphere"` (unit sphere) |
| `shapes.ReentrantCube` | `refinement_factor=0.2, element_size=0.1, label="reentrant_cube"` |
| `shapes.Almond` | `element_size=0.01, label="almond"` |

Useful methods and helpers:

- `geometry.number_of_vertices()` returns the mesh size, a quick proxy for run time.
- `geometry.export_mesh(filename)` writes a Gmsh `.msh` file.
- `optimus.geometry.load.import_grid(filename, label="imported_grid")` reads a mesh from file (for example from Gmsh).
- `optimus.utils.mesh.get_geometries_stats(geometries, verbose=True)` prints element size statistics.

In [ ]:
frequency = 500e3
wavelength = water.compute_wavelength(frequency)
sphere = optimus.geometry.shapes.Sphere(radius=5e-3, origin=(0, 0, 0), element_size=wavelength / 6)
print("Element size: %.3f mm, vertices: %d" % (1e3 * wavelength / 6, sphere.number_of_vertices()))
stats = optimus.utils.mesh.get_geometries_stats(sphere, verbose=True)

### 4.1 Moving and resizing meshes

All four helpers in `optimus.utils.mesh` return a **new** geometry and leave the original unchanged:

| Function | What it does |
|---|---|
| `scale_mesh(geometry, scaling_factor)` | Multiplies every coordinate by the factor, about the global origin. Use `1e-3` to convert a mesh drawn in mm to m. |
| `translate_mesh(geometry, translation_vector)` | Adds a 3-vector (in m) to every vertex |
| `rotate_mesh(geometry, rotation_angles)` | Rotates about the global origin; angles `(ax, ay, az)` in **radians**, applied about x, then y, then z |
| `rotate_mesh_around_center(geometry, rotation_angles)` | Same, but about the centre of the mesh |

In [ ]:
ellipsoid = optimus.geometry.shapes.Ellipsoid(radius=(2e-3, 1e-3, 1e-3), element_size=0.5e-3)
moved = optimus.utils.mesh.translate_mesh(ellipsoid, (10e-3, 0, 0))
turned = optimus.utils.mesh.rotate_mesh_around_center(moved, (0, 0, np.pi / 2))   # 90 degrees about z

for g in (ellipsoid, moved, turned):
    v = g.grid.leaf_view.vertices
    print("x range: %6.2f to %6.2f mm, y range: %6.2f to %6.2f mm"
          % (1e3 * v[0].min(), 1e3 * v[0].max(), 1e3 * v[1].min(), 1e3 * v[1].max()))

## 5. Materials (`optimus.material`)

```python
optimus.material.load_material(name)                  # name: str, or tuple/list of str
optimus.material.create_material(name, density, speed_of_sound,
                                 attenuation_coeff_a=0, attenuation_pow_b=0,
                                 heat_capacity=None, thermal_conductivity=None,
                                 save_to_file=False, **properties_user)
```

`load_material` reads the built-in database (names such as `"water"`, `"fat"`, `"bone (cortical)"`, `"air"`). Passing a tuple returns a list of materials in the same order. To see every available name, run `optimus.material.common.get_excel_database(database="default")`.

| `create_material` argument | Unit | Default |
|---|---|---|
| `density` | kg/m^3 | required |
| `speed_of_sound` | m/s | required |
| `attenuation_coeff_a` | Np/m/MHz^b | `0` (lossless) |
| `attenuation_pow_b` | dimensionless | `0` |
| `heat_capacity` | J/kg/C | `None` |
| `thermal_conductivity` | W/m/C | `None` |
| `save_to_file` | bool | `False`; `True` adds it to the user-defined database |

Methods on a material object:

| Method | Returns |
|---|---|
| `compute_wavelength(frequency)` | `speed_of_sound / frequency`, in m |
| `compute_attenuation(frequency)` | `a * (frequency / 1e6) ** b`, in Np/m |
| `compute_wavenumber(frequency)` | complex `2*pi*f/c + 1j*alpha`, in 1/m |
| `print()` | prints name, density, speed of sound and attenuation |

In [ ]:
fat, bone = optimus.material.load_material(("fat", "bone (cortical)"))
my_tissue = optimus.material.create_material(name="my-tissue", density=1050, speed_of_sound=1560,
                                             attenuation_coeff_a=5.0, attenuation_pow_b=1.1)

frequency = 1e6
for m in (water, fat, bone, my_tissue):
    print("%-16s wavelength = %.3f mm, attenuation = %.2f Np/m"
          % (m.name, 1e3 * m.compute_wavelength(frequency), m.compute_attenuation(frequency)))

## 6. Models (`optimus.model`)

```python
optimus.model.create_default_model(source, geometry, exterior, interior, label="default")
optimus.model.create_acoustic_model(source, geometry, exterior, interior,
                                    formulation="pmchwt", formulation_parameters=None,
                                    preconditioner="mass", preconditioner_parameters=None,
                                    label="custom")
```

- `create_default_model` is `create_acoustic_model` with the PMCHWT formulation and the mass-matrix preconditioner.
- For several separate scatterers, pass **lists** for `geometry` and `interior`, matched by position.
- `formulation`: only `"pmchwt"` is implemented for this model type.
- `preconditioner`: `"none"`, `"mass"` or `"osrc"` (OSRC suits higher frequencies, see `Preconditioners.ipynb`).

After `model.solve()` the useful attributes are:

| Attribute | Meaning |
|---|---|
| `model.iteration_count` | Number of GMRES iterations used |
| `model.solution` | Surface traces (bempp grid functions) |
| `model.frequency` | Frequency taken from the source |

In [ ]:
source = optimus.source.create_planewave(frequency=1000)
geometries = [optimus.geometry.shapes.Sphere(radius=0.5, origin=(-1, 0, 0), element_size=0.25),
              optimus.geometry.shapes.Sphere(radius=0.5, origin=(1, 0, 0), element_size=0.25)]
interiors = [fat, bone]

model_two = optimus.model.create_acoustic_model(source, geometries, water, interiors,
                                                formulation="pmchwt", preconditioner="mass")
model_two.solve()
print("Two scatterers solved in", model_two.iteration_count, "iterations")

### 6.1 Nested domains

For a scatterer inside another scatterer (for example a skull around brain tissue), describe the layout as a graph and use the nested model:

```python
graph = optimus.geometry.Graph(label)
graph.create_exterior_domain(material, source=None)
graph.create_interface(geometry, exterior_subdomain_id, interior_subdomain_material,
                       interior_interface_ids=None, source=None)
graph.print_graph_information()

optimus.model.create_default_nested_model(topology, frequency, label="default")
optimus.model.create_nested_model(topology, frequency, formulation, preconditioner,
                                  parameters=None, label="nested")
```

How the graph works:

- **Subdomains are numbered in the order they are created.** The exterior is subdomain 0, the inside of the first interface is subdomain 1, the inside of the second interface is subdomain 2, and so on.
- `exterior_subdomain_id` is the subdomain that **surrounds** the new surface. Two separate spheres side by side in water both use `0`; a sphere inside the first sphere uses `1`.
- `formulation` and `preconditioner` take one string for every interface, or a list with one entry per interface in creation order. Formulations: `"pmchwt"`, `"muller"`, `"multitrace"`. Preconditioners: `"none"`, `"mass"`, `"osrc"`, `"calderon"`.
- `create_default_nested_model` uses PMCHWT with the mass preconditioner on every interface.
- After `model.solve()`, the iteration count is in `model.linear_solve_iteration_count` (not `iteration_count`). Use `model.solve(timing=True)` to fill `model.timings`.

In [ ]:
frequency = 1000
source = optimus.source.create_planewave(frequency)

graph = optimus.geometry.Graph("fat_shell_with_bone_core")
graph.create_exterior_domain(water, source)

outer = optimus.geometry.shapes.Sphere(radius=1.0, element_size=0.3, label="outer")
inner = optimus.geometry.shapes.Sphere(radius=0.5, element_size=0.2, label="inner")
graph.create_interface(outer, 0, fat)    # surrounded by subdomain 0 (water), creates subdomain 1
graph.create_interface(inner, 1, bone)   # surrounded by subdomain 1 (fat), creates subdomain 2
graph.print_graph_information()

model_nested = optimus.model.create_nested_model(graph, frequency, "pmchwt", ["osrc", "mass"])
model_nested.solve()
print("Nested model solved in", model_nested.linear_solve_iteration_count, "iterations")

`Nested domains.ipynb` has a full example with three interfaces.

## 7. Global parameters (`optimus.global_parameters`)

Solver and assembly settings live in one global object. The ones changed most often:

| Setting | Default | Meaning |
|---|---|---|
| `optimus.global_parameters.linalg.tol` | `1e-5` | GMRES relative tolerance |
| `optimus.global_parameters.linalg.maxiter` | `1000` | Maximum GMRES iterations |
| `optimus.global_parameters.bem.set_matrix_assembly_type("dense")` | | Dense or H-matrix assembly of the BEM matrix |
| `optimus.global_parameters.bem.set_matrix_hmat(eps=...)` | | H-matrix accuracy |
| `optimus.global_parameters.verbosity` | `False` | Print progress information |

Use `optimus.global_parameters.print()` to see everything.

Less common settings:

| Setting | Default | Meaning |
|---|---|---|
| `optimus.global_parameters.bem.set_field_assembly_type("hmat")` | `"dense"` | Assembly of the potential operators used in postprocessing. H-matrix saves memory on large visualisation grids. |
| `optimus.global_parameters.bem.set_field_hmat(eps=...)` | `eps=1e-8` | H-matrix accuracy for postprocessing |
| `optimus.global_parameters.bem.set_quadrature_order(operator_type, region, order)` | | `operator_type`: `"boundary"`, `"potential"` or `"all"`; `region`: `"singular"`, `"near"`, `"medium"`, `"far"` or `"all"` |
| `optimus.global_parameters.incident_field_parallelisation.parallelisation_method` | `"numba"` | How transducer fields are computed: `"numba"`, `"multiprocessing"` or `"multiprocessing_shared"` |
| `optimus.global_parameters.incident_field_parallelisation.cpu_count` | all cores | Cores used by the multiprocessing methods |
| `optimus.global_parameters.preconditioning.osrc.npade` | `4` | Number of Padé terms in the OSRC preconditioner |
| `optimus.global_parameters.preconditioning.osrc.wavenumber` | `"int"` | Wavenumber used by OSRC: `"int"`, `"ext"` or a number |

Global parameters are read when `solve()` or `compute_fields()` runs, so set them before those calls. More detail in `Global parameters.ipynb`, `Preconditioners.ipynb` and `Incident field parallelisation.ipynb`.

## 8. Postprocessing (`optimus.postprocess`)

### 8.1 Field on a plane

```python
pp = optimus.postprocess.VisualisePlane(model, verbose=False)
pp.create_computational_grid(resolution=(141, 141), plane_axes=(0, 1),
                             plane_offset=0.0, bounding_box=None)
pp.compute_fields()
optimus.postprocess.plot_pressure_field(pp, field="total", unit="Pa",
                                        display_edges=True, clim=None, file_name=None)
```

| Argument | Meaning |
|---|---|
| `resolution` | Number of points along the two plane axes |
| `plane_axes` | Which axes span the plane: `(0, 1)` is the xy-plane, `(0, 2)` is xz |
| `plane_offset` | Position of the plane along the third axis, in m |
| `bounding_box` | `(h_min, h_max, v_min, v_max)` in m; `None` fits it to the geometry |
| `field` | `"total"`, `"scattered"` or `"incident"` |
| `unit` | `"Pa"`, `"kPa"`, `"MPa"` or `"GPa"` (plot scaling only) |
| `clim` | `(min, max)` colour limits, in the same unit as `unit` |

`plot_pressure_field` returns two matplotlib figures: the real part and the magnitude. After `compute_fields()`, the raw data is in `pp.total_field_imshow` (2D array ready for `imshow`) and `pp.field.total_field` (flat array matching `pp.points`); the same exists for `scattered` and `incident`.

In [ ]:
pp = optimus.postprocess.VisualisePlane(model)
pp.create_computational_grid(resolution=(81, 81), plane_axes=(0, 1), bounding_box=(-3, 3, -3, 3))
pp.compute_fields()
print("Max |p_total| on the plane: %.3f Pa" % np.nanmax(np.abs(pp.total_field_imshow)))
figs = optimus.postprocess.plot_pressure_field(pp, field="scattered", unit="Pa")
fig_regions = optimus.postprocess.plot_regions(pp)

### 8.2 Field at arbitrary points

`VisualiseCloudPoints` evaluates the field at any set of points, for example along a line. The results are in `cloud.field.total_field`, `cloud.field.scattered_field` and `cloud.field.incident_field`, in the same order as the points.

In [ ]:
x = np.linspace(-3, 3, 121)
line_points = np.vstack((x, np.zeros_like(x), np.zeros_like(x)))   # the x-axis, through the sphere

cloud = optimus.postprocess.VisualiseCloudPoints(model)
cloud.create_computational_grid(line_points)
cloud.compute_fields()

import matplotlib.pyplot as plt
plt.figure()
plt.plot(x, np.abs(cloud.field.total_field), label="total")
plt.plot(x, np.abs(cloud.field.incident_field), "--", label="incident")
plt.xlabel("x [m]")
plt.ylabel("|p| [Pa]")
plt.legend()
plt.show()

### 8.3 Other visualisation classes

| Class | Use it for | Main calls | Example notebook |
|---|---|---|---|
| `VisualisePlaneAndBoundary(model)` | Field on a plane **and** on the scatterer surface, written to Gmsh files | `create_computational_grid(...)` (same arguments as `VisualisePlane`), `compute_fields(file_name="planar_and_surface")` | `Visualisation and postprocess.ipynb` |
| `Visualise3DGrid(geometries)` | Interactive 3D view of one or more meshes (needs `k3d`) | `create_computational_grid()`, `display_field()` | `Visualisation in 3D.ipynb` |
| `Visualise3DField(model)` | Interactive 3D view of the surface field plus planes | `create_computational_grid()`, `add_VisualisePlane(pp)`, `compute_fields()`, `display_field(surface=True, planes=True)` | `Visualisation in 3D.ipynb` |
| `VisualiseTimeDomain(model)` | Animation of the harmonic field over time on an existing plane | `create_computational_grid(time_length=1, n_samples=100)` (length in periods), `compute_fields(pp)`, `display_field()` | `Visualisation and postprocess.ipynb` |

For broadband signals (rather than animating one frequency), see `Time domain simulations.ipynb`, which solves at many frequencies and combines them with an FFT.

## 9. Export and import (`optimus.postprocess`)

```python
optimus.postprocess.export_to_file(model, post_process, global_parameters,
                                   file_name="optimus_exported_data", file_format="db")
data = optimus.postprocess.import_from_file("optimus_exported_data.db")
```

- `file_format="db"` (default) uses Python's `shelve` module. The imported dictionary has the keys `"global_parameters"`, `"source"`, `"model"` and `"post_process"`; the large BEM operators are stripped from the model before saving.
- `file_format="mat"` writes a MATLAB file with `total_field`, `scattered_field`, `incident_field`, `points`, `L2_norm_ptot_MPa` and `gmres_iter_count`. In v0.2.1 it reads `post_process.total_field`, which only `VisualisePlaneAndBoundary` sets, so with `VisualisePlane` it raises an `AttributeError`. Use `"db"`, or save the arrays yourself as in the second cell below.
- `export_to_file` expects a single-interface model from `create_default_model` or `create_acoustic_model`; it does not handle nested models.

In [ ]:
optimus.postprocess.export_to_file(model, pp, optimus.global_parameters,
                                   file_name="quickref_example", file_format="db")
data = optimus.postprocess.import_from_file("quickref_example.db")
print(sorted(data.keys()))
print("Frequency stored with the source:", data["source"].frequency, "Hz")

In [ ]:
# MATLAB export for a VisualisePlane result
import scipy.io
scipy.io.savemat("quickref_plane.mat", {"points": pp.points,
                                        "total_field": pp.field.total_field,
                                        "incident_field": pp.field.incident_field,
                                        "scattered_field": pp.field.scattered_field})

## 10. Common errors and fixes

| Symptom | Likely cause | Fix |
|---|---|---|
| Meshing or solving takes forever, or memory runs out | Lengths given in mm instead of m (for example `radius=32` instead of `32e-3`), so the mesh is 1000 times too fine relative to the object | Use metres everywhere; check `geometry.number_of_vertices()` before solving |
| Field looks noisy or wrong, but there is no error | Mesh too coarse. OptimUS does **not** check elements per wavelength. | Set `element_size = wavelength / 6` (or finer) using the shortest wavelength in any material |
| `RuntimeWarning: The GMRES solver stopped at the maximum number of ... iterations` | The solver did not converge | Use `preconditioner="osrc"`, raise `optimus.global_parameters.linalg.maxiter`, or refine the mesh |
| `ValueError: the material: ... is not in the database.` | Misspelt name (the lookup ignores case, but not spelling or spaces) | Check the names with `optimus.material.common.get_excel_database()` |
| `ValueError: ... needs to be three dimensional.` | A points array that is neither (3, N) nor (N, 3) | Build points with `np.vstack((x, y, z))`. Note that (N, 3) is transposed automatically, so a 3x3 array is always read as columns = points. |
| `ValueError: Array element centroid locations do not appear to lie on a sphere.` | Spherical array centroids not all at the same distance from the local origin | Give centroids relative to the focus (see section 3.5), or use `array_type="planar"` |
| `ValueError: The number of preconditioners must be equal to the number of interfaces.` | Nested model list with the wrong length | One entry per `create_interface` call, in creation order |
| `AttributeError` from `export_to_file(..., file_format="mat")` | Known limitation with `VisualisePlane` | See section 9 |
| `ImportError: No module named optimus` | Running `python` (Python 2.7) instead of `python3` inside the Docker image | Use `python3`, or the Jupyter Python 3 kernel |